[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap05/cap05.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

A presente lista de exercícios de programação (EP) consolida as formulações teóricas apresentadas ao longo do Capítulo 5 — Transformadas e Compressão — por meio de uma trilha prática aplicada. Os exercícios são estruturados a partir de matrizes de dimensões reduzidas, viabilizando a validação analítica e a inspeção manual de cada coeficiente, mantendo a consistência metodológica adotada nos capítulos anteriores.

O encadeamento dos exercícios reproduz rigorosamente o fluxo conceitual do capítulo: inicia-se com a manipulação direta de espectros previamente computados (projeto de filtros passa-baixa e máscaras *notch*); avança-se para o processo de quantização de coeficientes, que constitui o núcleo da compressão com perda; desenvolve-se a implementação explícita da Transformada Discreta de Fourier (DFT) a partir de sua definição matemática fundamental; e conclui-se com a integração dessas etapas na construção de um *pipeline* de compressão JPEG simplificado.

::: {.callout-important}
### Diretrizes para a Resolução dos Exercícios de Programação {.unnumbered}

Em todos os exercícios deste capítulo, as coordenadas do **centro do espectro** (origem das frequências espaciais pós-aplicação do deslocamento `fftshift`) devem ser determinadas via divisão inteira. Para uma matriz com $L$ linhas e $C$ colunas, a componente de frequência nula localiza-se na posição:

$$(c_y, c_x) = \left( \left\lfloor \frac{L}{2} \right\rfloor, \left\lfloor \frac{C}{2} \right\rfloor \right)$$

Esta convenção é rigorosamente idêntica à adotada pela função `np.fft.fftshift`. Ademais, em todas as etapas que exijam discretização ou arredondamento numérico (seja na quantização de coeficientes AC ou na reconstrução final de pixels), deve-se empregar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*), mitigando ambiguidades em valores com fração exatamente igual a $0.5$.
:::

### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### *Download* {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [64]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.2 | TestSuite: 1.1.2


#### Executando os Testes {.unnumbered}
Para rodar os testes, execute `TestSuite("EP05_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como string numa variável `codigo`:

```python
codigo = """
from morph import mm
# ... seu código aqui ...
"""
TestSuite("EP05_01").run_code(codigo)
```

### EP05_01 🟢 Filtro Passa-Baixa Ideal por Distância no Espectro

Em um **scanner de documentos antigo**, o sensor capta papel amassado e textura de fibra junto com o texto — ruído de alta frequência que "polui" o espectro nas bordas. O técnico de manutenção não tem acesso à imagem original, apenas ao **espectro de magnitude já calculado** pelo software do scanner. Seu trabalho é simples e cirúrgico: manter apenas o **círculo central** de baixas frequências (a estrutura global do documento) e apagar tudo que estiver fora do raio $D_0$, eliminando a textura fina sem nem precisar tocar na imagem espacial.

Este é o **Filtro Passa-Baixa Ideal (LPFI)**: a operação espectral mais direta do capítulo, mas também a que melhor revela a anatomia de um espectro centrado.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas) do espectro de magnitude — já fornecido **centrado** (equivalente à saída de `np.fft.fftshift`).
2. **Frequência de corte:** Ler o inteiro $D_0$.
3. **Dados:** Ler os valores inteiros da matriz de magnitude, linha a linha.
4. **Centro do espectro:** Calcular $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
5. **Distância:** Para cada posição $(u,v)$, calcular
$$
D(u,v) = \sqrt{(u-c_y)^2 + (v-c_x)^2}
$$
6. **Máscara ideal:** Aplicar
$$
H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}
$$
7. **Filtragem:** O valor de saída é $\text{mag}'(u,v) = \text{mag}(u,v) \cdot H(u,v)$.
8. **Saída:** Exibir a matriz filtrada com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Comparação não estrita:** o critério usa $D(u,v) \le D_0$ (a fronteira pertence ao filtro, ou seja, é mantida).
* **Tipo:** todos os valores de entrada e saída são inteiros; a distância é calculada em ponto flutuante apenas internamente.
* **Sem arredondamento de magnitude:** como a entrada já é inteira e a máscara é binária (0 ou 1), a saída nunca precisa de arredondamento.

#### 🧠 Fundamentação Teórica

| Região | Distância ao centro | Efeito do filtro |
|---|---|---|
| **Centro** ($D \le D_0$) | Baixas frequências | Preservadas — estrutura global mantida |
| **Bordas** ($D > D_0$) | Altas frequências | Zeradas — textura e ruído removidos |
| **$D_0$ pequeno** | — | Imagem reconstruída ficaria muito borrada |
| **$D_0$ grande** | — | Pouca filtragem; quase toda energia preservada |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $D_0$.
* Linhas seguintes: Elementos inteiros da matriz de magnitude (centrada).

**Saída:**

* Matriz filtrada em $L$ linhas e $C$ colunas, separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>1<br>10 20 30<br>40 50 60<br>70 80 90 | 0 20 0<br>40 50 60<br>0 80 0 | Centro $(1,1)$. Cantos têm $D=\sqrt{2}\approx1.41 > 1$, logo são zerados; vizinhos ortogonais têm $D=1 \le 1$ e são mantidos. |
| 1<br>3<br>0<br>5 9 7 | 0 9 0 | $L=1, C=3$: centro em $(0,1)$. Apenas a própria posição central ($D=0$) sobrevive a $D_0=0$. |



In [65]:
#| label: fig-05-sim-ep01
#| fig-cap: "Simulador: Filtro Passa-Baixa Ideal no Espectro"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0501" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Filtro Passa-Baixa Ideal</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 H = (D ≤ D₀) ? 1 : 0</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste D₀ e observe quais posições do espectro 5×5 sobrevivem ao filtro.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">D₀ (raio de corte)</label>
        <span id="ep0501_vl_d0" style="font-family:monospace;font-weight:bold;color:#2980b9;">1</span>
      </div>
      <input id="ep0501_sl_d0" style="width:100%;accent-color:#2980b9;" max="4" min="0" step="1" type="range" value="1">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Espectro Original (magnitude)</p>
        <div id="ep0501_grid_orig" style="display:grid;grid-template-columns:repeat(5,42px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Resultado Filtrado</p>
        <div id="ep0501_grid_new" style="display:grid;grid-template-columns:repeat(5,42px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0501_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.init) return;
    root.dataset.init = "1";
    var N = 5, cy = Math.floor(N/2), cx = Math.floor(N/2);
    var mag = [];
    for(var i=0;i<N;i++){ var row=[]; for(var j=0;j<N;j++){ row.push(10*(i+1)+j+1); } mag.push(row); }
    var d0el = root.querySelector('#ep0501_sl_d0');
    var d0v  = root.querySelector('#ep0501_vl_d0');
    var go = root.querySelector('#ep0501_grid_orig');
    var gn = root.querySelector('#ep0501_grid_new');
    var dbg = root.querySelector('#ep0501_debug');

    function cellStyle(active){
      return 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;' +
             (active ? 'background:#dbeafe;color:#1e40af;border:1px solid #93c5fd;' : 'background:#f3f4f6;color:#9ca3af;border:1px solid #e5e7eb;');
    }

    function render(){
      var D0 = parseInt(d0el.value);
      d0v.textContent = D0;
      go.innerHTML=''; gn.innerHTML='';
      var kept = 0;
      for(var i=0;i<N;i++){
        for(var j=0;j<N;j++){
          var d = Math.sqrt((i-cy)*(i-cy)+(j-cx)*(j-cx));
          var keep = d <= D0;
          if(keep) kept++;
          var co = document.createElement('div'); co.style.cssText = cellStyle(true); co.textContent = mag[i][j];
          go.appendChild(co);
          var cn = document.createElement('div'); cn.style.cssText = cellStyle(keep); cn.textContent = keep ? mag[i][j] : 0;
          gn.appendChild(cn);
        }
      }
      dbg.textContent = 'Centro=(' + cy + ',' + cx + ')  |  D₀=' + D0 + '  |  Coeficientes mantidos: ' + kept + '/' + (N*N);
    }
    d0el.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0501');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

In [66]:
%%writefile EP05_01.py
# Código Python

Overwriting EP05_01.py


In [67]:
TestSuite("EP05_01.py").run()

### EP05_02 🟡 Filtro Notch: Removendo Picos Periódicos

Uma câmera de **inspeção industrial** captura imagens de placas de circuito, mas a fonte de alimentação da linha de produção introduz uma **interferência elétrica periódica** — um padrão de listras quase imperceptível a olho nu, mas que aparece no espectro de Fourier como **pares de picos brilhantes** simetricamente posicionados em torno do centro. A equipe de visão computacional não pode reprocessar a captura: precisa **localizar e apagar cirurgicamente** esses pares de picos no espectro, preservando todo o resto da informação útil da imagem.

Esse é o papel do **filtro rejeita-banda notch**: diferente do passa-baixa (que afeta uma região contínua), ele ataca **pontos específicos e seus simétricos**, deixando o restante do espectro intocado.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas) do espectro de magnitude centrado.
2. **Dados:** Ler os valores inteiros da matriz de magnitude, linha a linha.
3. **Picos:** Ler o inteiro $K$ (quantidade de pares de picos a remover).
4. **Para cada um dos $K$ picos:** ler três inteiros $\Delta v$, $\Delta u$, $r$ — deslocamento vertical, deslocamento horizontal e raio do notch.
5. **Centro do espectro:** $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
6. **Supressão simétrica:** para cada pico, zerar **todas** as posições $(u,v)$ tais que a distância ao ponto $(c_y+\Delta v,\, c_x+\Delta u)$ for $\le r$, **e também** todas as posições com distância $\le r$ ao ponto simétrico $(c_y-\Delta v,\, c_x-\Delta u)$.
7. **Saída:** Exibir a matriz resultante com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Simetria obrigatória:** cada pico informado gera **dois** discos zerados (o ponto e seu simétrico em relação ao centro) — esquecer o simétrico é o erro mais comum.
* **Sobreposição:** se dois discos se sobrepõem, a posição permanece zerada (não há "soma" ou restauração).
* **Comparação não estrita:** uma posição é zerada se $\text{distância} \le r$.
* **Ordem de leitura:** os $K$ picos devem ser processados na ordem em que aparecem na entrada, mas o resultado final independe da ordem (operações de zerar são comutativas).

#### 🧠 Fundamentação Teórica

| Conceito | Papel no filtro notch |
|---|---|
| **Pico em $(\Delta v, \Delta u)$** | Frequência da interferência periódica detectada visualmente no espectro |
| **Ponto simétrico $(-\Delta v,-\Delta u)$** | Toda DFT de sinal real é hermitiana: picos sempre aparecem em pares simétricos ao centro |
| **Raio $r$** | Controla a "largura" da rejeição — $r$ grande remove mais energia ao redor do pico, mas também informação útil |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos inteiros da matriz de magnitude (centrada), $L$ linhas.
* Próxima linha: Inteiro $K$.
* $K$ linhas seguintes: três inteiros $\Delta v$, $\Delta u$, $r$ (separados por espaço).

**Saída:**

* Matriz resultante em $L$ linhas e $C$ colunas, separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5<br>5<br>1 2 3 4 5<br>6 7 8 9 10<br>11 12 13 14 15<br>16 17 18 19 20<br>21 22 23 24 25<br>1<br>1 1 0 | 1 2 3 4 5<br>6 0 8 9 10<br>11 12 13 14 15<br>16 17 18 0 20<br>21 22 23 24 25 | Centro $(2,2)$. Pico em $(\Delta v,\Delta u)=(1,1) \to (3,3)$ [valor 19] e simétrico $(-1,-1) \to (1,1)$ [valor 7] são zerados ($r=0$, apenas o próprio ponto). |



In [68]:
#| label: fig-05-sim-ep02
#| fig-cap: "Simulador: Filtro Notch"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0502" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Filtro Notch</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 par simétrico</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Mova Δv e Δu para escolher o pico — note como o par simétrico também é apagado.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;display:flex;gap:16px;flex-wrap:wrap;">
      <div style="flex:1;min-width:140px;">
        <div style="display:flex;justify-content:space-between;"><label style="font-size:12px;font-weight:bold;color:#b8860b;">Δv</label><span id="ep0502_vl_dv" style="font-family:monospace;font-weight:bold;">1</span></div>
        <input id="ep0502_sl_dv" style="width:100%;accent-color:#b8860b;" max="2" min="-2" step="1" type="range" value="1">
      </div>
      <div style="flex:1;min-width:140px;">
        <div style="display:flex;justify-content:space-between;"><label style="font-size:12px;font-weight:bold;color:#b8860b;">Δu</label><span id="ep0502_vl_du" style="font-family:monospace;font-weight:bold;">1</span></div>
        <input id="ep0502_sl_du" style="width:100%;accent-color:#b8860b;" max="2" min="-2" step="1" type="range" value="1">
      </div>
      <div style="flex:1;min-width:140px;">
        <div style="display:flex;justify-content:space-between;"><label style="font-size:12px;font-weight:bold;color:#b8860b;">r</label><span id="ep0502_vl_r" style="font-family:monospace;font-weight:bold;">0</span></div>
        <input id="ep0502_sl_r" style="width:100%;accent-color:#b8860b;" max="2" min="0" step="1" type="range" value="0">
      </div>
    </div>
    <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
      <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Espectro 5×5 (vermelho = removido)</p>
      <div id="ep0502_grid" style="display:grid;grid-template-columns:repeat(5,42px);gap:4px;justify-content:center;"></div>
    </div>
    <div id="ep0502_debug" style="margin-top:20px;background:#fdf6e3;border-radius:8px;padding:10px;border:1px solid #f5e7c1;font-family:monospace;font-size:11px;color:#8a6d1d;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.init) return;
    root.dataset.init = "1";
    var N=5, cy=Math.floor(N/2), cx=Math.floor(N/2);
    var mag=[]; for(var i=0;i<N;i++){var row=[];for(var j=0;j<N;j++){row.push(i*5+j+1);} mag.push(row);}
    var dv=root.querySelector('#ep0502_sl_dv'), du=root.querySelector('#ep0502_sl_du'), r=root.querySelector('#ep0502_sl_r');
    var dvv=root.querySelector('#ep0502_vl_dv'), duv=root.querySelector('#ep0502_vl_du'), rv=root.querySelector('#ep0502_vl_r');
    var grid=root.querySelector('#ep0502_grid'), dbg=root.querySelector('#ep0502_debug');
    function render(){
      var DV=parseInt(dv.value), DU=parseInt(du.value), R=parseInt(r.value);
      dvv.textContent=DV; duv.textContent=DU; rv.textContent=R;
      var p1=[cy+DV, cx+DU], p2=[cy-DV, cx-DU];
      grid.innerHTML=''; var removed=0;
      for(var i=0;i<N;i++){
        for(var j=0;j<N;j++){
          var d1=Math.sqrt((i-p1[0])*(i-p1[0])+(j-p1[1])*(j-p1[1]));
          var d2=Math.sqrt((i-p2[0])*(i-p2[0])+(j-p2[1])*(j-p2[1]));
          var kill = (d1<=R) || (d2<=R);
          if(kill) removed++;
          var c=document.createElement('div');
          c.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;' +
            (kill ? 'background:#fee2e2;color:#b91c1c;border:1px solid #fca5a5;' : 'background:#f3f4f6;color:#374151;border:1px solid #e5e7eb;');
          c.textContent = kill ? 0 : mag[i][j];
          grid.appendChild(c);
        }
      }
      dbg.textContent = 'Centro=(' + cy + ',' + cx + ')  |  Pico=(' + p1[0] + ',' + p1[1] + ')  |  Simétrico=(' + p2[0] + ',' + p2[1] + ')  |  Removidos: ' + removed;
    }
    [dv,du,r].forEach(function(el){ el.addEventListener('input', render); });
    render();
  }
  function tryInit(){ var root=document.getElementById('sim-ep0502'); if(root) init(root); else setTimeout(tryInit,200); }
  tryInit();
})();
</script>
''')

In [69]:
%%writefile EP05_02.py
# Código Python

Writing EP05_02.py


In [70]:
TestSuite("EP05_02.py").run()

### EP05_03 🟠 Quantização DCT: a Verdadeira Fonte de Compressão

Um aplicativo de **galeria de fotos** precisa reduzir o tamanho de milhares de imagens antes de fazer *upload* para a nuvem, sem recodificar tudo do zero. O engenheiro responsável já tem os **coeficientes DCT** de cada bloco $4\times4$ calculados (a etapa cara computacionalmente já foi feita) — falta apenas aplicar a **tabela de quantização**, a etapa que realmente descarta informação e gera compressão. Coeficientes de alta frequência, menos perceptíveis ao olho humano, recebem divisores grandes e tendem a virar **zero**; coeficientes de baixa frequência, mais perceptíveis, recebem divisores pequenos e sobrevivem quase intactos.

Você vai implementar exatamente essa etapa: **quantizar e desquantizar** (dividir, arredondar, multiplicar de volta) — o coração da compressão *lossy* do JPEG.

#### 📋 Diretrizes de Implementação

1. **Dimensão do bloco:** Ler o inteiro $N$ (bloco $N \times N$).
2. **Coeficientes:** Ler a matriz $C$ de coeficientes DCT, $N$ linhas com $N$ inteiros cada (podem ser negativos).
3. **Tabela de quantização:** Ler a matriz $Q$, $N$ linhas com $N$ inteiros positivos cada.
4. **Quantização:** Para cada posição $(u,v)$, calcular o índice quantizado
$$
\tilde{C}(u,v) = \text{round}\!\left(\frac{C(u,v)}{Q(u,v)}\right)
$$
usando arredondamento padrão para o inteiro mais próximo (valores intermediários `.5` nunca ocorrem nos casos de teste).
5. **Desquantização (reconstrução):** Calcular
$$
C'(u,v) = \tilde{C}(u,v) \times Q(u,v)
$$
6. **Saída:** Exibir a matriz reconstruída $C'$, $N \times N$, inteiros.

#### 📌 Restrições Computacionais

* **Round-trip completo:** a saída é o coeficiente **reconstruído** ($\tilde{C} \times Q$), não o índice quantizado isolado.
* **Divisão em ponto flutuante:** a divisão $C(u,v)/Q(u,v)$ deve ser feita em ponto flutuante antes do arredondamento — divisão inteira truncada produzirá resultado incorreto.
* **Sinal preservado:** coeficientes negativos mantêm o sinal após quantização e reconstrução.
* **$Q(u,v) > 0$ sempre:** não há necessidade de tratar divisão por zero.

#### 🧠 Fundamentação Teórica

| Coeficiente | Frequência | Valor típico de $Q$ | Efeito da quantização |
|---|---|---|---|
| $C(0,0)$ | DC (média do bloco) | Pequeno | Quase sempre sobrevive — domina a energia |
| $C(u,v)$ baixo $u+v$ | Baixa frequência | Pequeno/médio | Parcialmente preservado |
| $C(u,v)$ alto $u+v$ | Alta frequência | Grande | Frequentemente vira zero — fonte da compressão |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$.
* $N$ linhas seguintes: matriz $C$ (coeficientes DCT, inteiros, podem ser negativos).
* $N$ linhas seguintes: matriz $Q$ (tabela de quantização, inteiros positivos).

**Saída:**

* Matriz reconstruída $C'$, $N \times N$, inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 4<br>50 10 -5 0<br>8 -3 2 1<br>0 1 0 0<br>2 0 0 -1<br>2 5 7 8<br>4 7 8 11<br>6 8 11 12<br>9 11 12 14 | 50 10 -7 0<br>8 0 0 0<br>0 0 0 0<br>0 0 0 0 | $C(0,0)=50/2=25 \to 25\times2=50$ (preservado). $C(0,2)=-5/7\approx-0.71\to-1\to-1\times7=-7$. Já $C(1,1)=-3/7\approx-0.43\to0$: zerado pela quantização — a maior parte do bloco vira zero, ilustrando a compactação de energia no canto superior esquerdo. |



In [71]:
#| label: fig-05-sim-ep03
#| fig-cap: "Simulador: Quantização DCT (round-trip)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0503" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Quantização DCT</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟠 round(C/Q)×Q</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste a escala de Q e veja quantos coeficientes sobrevivem (não-zero) após o round-trip.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#c2620a;">Escala de Q (agressividade)</label>
        <span id="ep0503_vl_s" style="font-family:monospace;font-weight:bold;color:#c2620a;">1.0×</span>
      </div>
      <input id="ep0503_sl_s" style="width:100%;accent-color:#c2620a;" max="4" min="0.25" step="0.25" type="range" value="1">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Coeficientes DCT (C)</p>
        <div id="ep0503_grid_c" style="display:grid;grid-template-columns:repeat(4,46px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Reconstruído (round(C/Q)·Q)</p>
        <div id="ep0503_grid_r" style="display:grid;grid-template-columns:repeat(4,46px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0503_debug" style="margin-top:20px;background:#fff1e6;border-radius:8px;padding:10px;border:1px solid #fcd9b8;font-family:monospace;font-size:11px;color:#9a4b0c;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.init) return;
    root.dataset.init = "1";
    var C=[[50,10,-5,0],[8,-3,2,1],[0,1,0,0],[2,0,0,-1]];
    var Qbase=[[2,5,7,8],[4,7,8,11],[6,8,11,12],[9,11,12,14]];
    var s = root.querySelector('#ep0503_sl_s');
    var sv = root.querySelector('#ep0503_vl_s');
    var gc = root.querySelector('#ep0503_grid_c');
    var gr = root.querySelector('#ep0503_grid_r');
    var dbg = root.querySelector('#ep0503_debug');
    function cell(v, faded){
      var c=document.createElement('div');
      c.style.cssText='width:46px;height:36px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;' +
        (faded ? 'background:#f3f4f6;color:#9ca3af;border:1px solid #e5e7eb;' : 'background:#fef3e2;color:#9a4b0c;border:1px solid #fcd9b8;');
      c.textContent=v;
      return c;
    }
    function render(){
      var scale=parseFloat(s.value);
      sv.textContent=scale.toFixed(2)+'×';
      gc.innerHTML=''; gr.innerHTML='';
      var zeros=0, total=16;
      for(var i=0;i<4;i++){
        for(var j=0;j<4;j++){
          gc.appendChild(cell(C[i][j], false));
          var Q = Qbase[i][j]*scale;
          var q = Math.round(C[i][j]/Q);
          var rec = Math.round(q*Q);
          if(rec===0) zeros++;
          gr.appendChild(cell(rec, rec===0));
        }
      }
      dbg.textContent = 'Zeros: ' + zeros + '/' + total + '  |  Quanto maior a escala de Q, mais zeros — maior compressão, menor qualidade.';
    }
    s.addEventListener('input', render);
    render();
  }
  function tryInit(){ var root=document.getElementById('sim-ep0503'); if(root) init(root); else setTimeout(tryInit,200); }
  tryInit();
})();
</script>
''')

In [72]:
%%writefile EP05_03.py
# Código Python

Writing EP05_03.py


In [73]:
TestSuite("EP05_03.py").run()

### EP05_04 🔴 Implementando a DFT 2D a Partir da Definição

Um laboratório de pesquisa em **astronomia computacional** recebeu, de uma missão antiga, um pequeno sensor experimental cujos dados brutos não podem ser processados por bibliotecas modernas de FFT — o ambiente de validação é isolado e só permite operações aritméticas básicas. A equipe precisa **reimplementar a Transformada de Fourier Discreta 2D a partir da própria definição matemática**, célula por célula, para depois comparar bit a bit com `np.fft.fft2` em outro ambiente.

Este é o exercício mais conceitual da lista: não há atalhos. Você vai implementar o duplo somatório da @eq-05-dft diretamente, evidenciando *por que* a FFT existe — e o custo computacional que ela evita.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $M$ (linhas) e $N$ (colunas) da imagem $f(x,y)$.
2. **Dados:** Ler os valores inteiros de $f(x,y)$, linha a linha.
3. **DFT 2D:** Para cada par de frequências $(u,v)$ com $u=0,\ldots,M-1$ e $v=0,\ldots,N-1$, calcular
$$
F(u,v) = \sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$
usando a identidade de Euler $e^{-j\theta} = \cos(\theta) - j\sin(\theta)$ para separar parte real e imaginária — **não utilize nenhuma função de FFT pronta**.
4. **Magnitude:** Calcular $|F(u,v)| = \sqrt{\text{Re}(F)^2 + \text{Im}(F)^2}$ e arredondar para o inteiro mais próximo.
5. **Saída:** Exibir a matriz de magnitudes arredondadas, $M \times N$, na mesma ordem (sem `fftshift` — o DC permanece em $(0,0)$).

#### 📌 Restrições Computacionais

* **Proibido usar bibliotecas de FFT:** a implementação deve calcular os somatórios duplos explicitamente (laços aninhados), mesmo que mais lenta.
* **Sem `fftshift`:** a saída mantém a convenção crua da DFT, com o componente DC em $F(0,0)$ (canto superior esquerdo).
* **Arredondamento:** a magnitude final deve ser arredondada para o inteiro mais próximo; nos casos de teste não há ambiguidade `.5`.
* **Precisão:** pequenos erros de ponto flutuante (ordem de $10^{-6}$) antes do arredondamento são esperados e não afetam o resultado inteiro final.

#### 🧠 Fundamentação Teórica

| Elemento | Significado |
|---|---|
| $F(0,0)$ | Componente DC — soma de todos os pixels, $F(0,0) = \sum f(x,y)$ |
| Parte real $\text{Re}(F)$ | Projeção do sinal sobre cossenos |
| Parte imaginária $\text{Im}(F)$ | Projeção do sinal sobre senos |
| Complexidade desta implementação | $\mathcal{O}((MN)^2)$ — por isso a FFT, com $\mathcal{O}(MN\log(MN))$, é indispensável em imagens reais |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $M$.
* Linha 2: Inteiro $N$.
* Linhas seguintes: Elementos inteiros de $f(x,y)$, $M$ linhas.

**Saída:**

* Matriz de magnitudes $|F(u,v)|$ arredondadas, $M \times N$, separadas por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 2<br>2<br>1 2<br>3 4 | 10 2<br>4 0 | $F(0,0)=1+2+3+4=10$ (DC = soma total). $F(0,1)=(1-2)+(3-4)=-2 \to |F|=2$. $F(1,0)=(1+2)-(3+4)=-4\to|F|=4$. $F(1,1)=(1-2)-(3-4)=0$. |



In [74]:
#| label: fig-05-sim-ep04
#| fig-cap: "Simulador: DFT 2D manual"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0504" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: DFT 2D — Definição Direta</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 ΣΣ f(x,y)e⁻ʲ²ᵖ(…)</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Clique nas células de f(x,y) para alterar os valores (incrementa de 1 em 1, shift+clique decrementa) e veja F(u,v) recalculado ao vivo.</p>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">f(x,y) — domínio espacial</p>
        <div id="ep0504_grid_f" style="display:grid;grid-template-columns:repeat(2,52px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">|F(u,v)| — magnitude (sem shift)</p>
        <div id="ep0504_grid_F" style="display:grid;grid-template-columns:repeat(2,52px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0504_debug" style="margin-top:20px;background:#fde8e8;border-radius:8px;padding:10px;border:1px solid #f8c9c9;font-family:monospace;font-size:11px;color:#9b1c1c;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.init) return;
    root.dataset.init = "1";
    var f=[[1,2],[3,4]];
    var gf=root.querySelector('#ep0504_grid_f'), gF=root.querySelector('#ep0504_grid_F'), dbg=root.querySelector('#ep0504_debug');
    function render(){
      gf.innerHTML=''; gF.innerHTML='';
      for(var x=0;x<2;x++){
        for(var y=0;y<2;y++){
          (function(xx,yy){
            var c=document.createElement('div');
            c.style.cssText='width:52px;height:42px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:13px;font-weight:bold;font-family:monospace;cursor:pointer;background:#dbeafe;color:#1e40af;border:1px solid #93c5fd;';
            c.textContent=f[xx][yy];
            c.addEventListener('click', function(e){
              if(e.shiftKey){ f[xx][yy]--; } else { f[xx][yy]++; }
              render();
            });
            gf.appendChild(c);
          })(x,y);
        }
      }
      var M=2,N=2;
      for(var u=0;u<M;u++){
        for(var v=0;v<N;v++){
          var re=0, im=0;
          for(var x=0;x<M;x++){
            for(var y=0;y<N;y++){
              var theta = 2*Math.PI*(u*x/M + v*y/N);
              re += f[x][y]*Math.cos(theta);
              im -= f[x][y]*Math.sin(theta);
            }
          }
          var mag = Math.round(Math.sqrt(re*re+im*im));
          var c=document.createElement('div');
          c.style.cssText='width:52px;height:42px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:13px;font-weight:bold;font-family:monospace;background:#fee2e2;color:#991b1b;border:1px solid #fca5a5;';
          c.textContent=mag;
          gF.appendChild(c);
        }
      }
      dbg.textContent = 'F(0,0) = soma de todos os pixels = ' + (f[0][0]+f[0][1]+f[1][0]+f[1][1]) + ' (componente DC)';
    }
    render();
  }
  function tryInit(){ var root=document.getElementById('sim-ep0504'); if(root) init(root); else setTimeout(tryInit,200); }
  tryInit();
})();
</script>
''')

In [75]:
%%writefile EP05_04.py
# Código Python

Writing EP05_04.py


In [76]:
TestSuite("EP05_04.py").run()

### EP05_05 🏆 Pipeline JPEG Completo: DCT, Quantização e Reconstrução

Você foi contratado para criar, do zero, um **codec JPEG didático** em ambiente embarcado, sem qualquer biblioteca de imagem disponível — apenas operações matemáticas básicas. O cliente quer entender exatamente onde a qualidade é perdida e onde ela é recuperada, bloco por bloco. Este é o desafio final do capítulo: integrar **tudo** o que foi estudado — a DCT-II ortonormal, a quantização perceptual e a reconstrução via IDCT — em um único pipeline de ponta a ponta, processando um bloco $N \times N$ do início ao fim, exatamente como o padrão JPEG faz internamente, $8\times8$ pixels de cada vez.

#### 📋 Diretrizes de Implementação

1. **Dimensão do bloco:** Ler o inteiro $N$.
2. **Bloco original:** Ler a matriz de pixels $f(x,y)$, $N$ linhas com $N$ inteiros em $[0,255]$.
3. **Tabela de quantização:** Ler a matriz $Q$, $N \times N$ inteiros positivos.
4. **Centralização:** Subtrair 128 de cada pixel: $g(x,y) = f(x,y) - 128$.
5. **DCT-II 2D ortonormal:** Calcular
$$
C(u,v) = \alpha(u)\,\alpha(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} g(x,y)\,\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]\cos\!\left[\frac{\pi(2y+1)v}{2N}\right]
$$
com $\alpha(0)=\sqrt{1/N}$ e $\alpha(k)=\sqrt{2/N}$ para $k>0$.
6. **Quantização:** $\tilde{C}(u,v) = \text{round}(C(u,v)/Q(u,v))$.
7. **Desquantização:** $C'(u,v) = \tilde{C}(u,v)\times Q(u,v)$.
8. **IDCT-II 2D (inversa ortonormal):** Calcular $g'(x,y)$ a partir de $C'(u,v)$ usando a transformada inversa correspondente (mesma base, somatório sobre $u,v$).
9. **Reversão da centralização e arredondamento:** $f'(x,y) = \text{round}(g'(x,y) + 128)$, restrito ao intervalo $[0,255]$ (*clipping*).
10. **Saída:** Exibir o bloco reconstruído $f'$, $N \times N$, inteiros.

#### 📌 Restrições Computacionais

* **Pipeline completo obrigatório:** todas as seis etapas (centralizar, DCT, quantizar, desquantizar, IDCT, reverter) devem ser implementadas — pular a quantização não passa nos testes, pois o resultado seria idêntico ao original.
* **Clipping:** valores reconstruídos fora de $[0,255]$ devem ser truncados (0 se negativo, 255 se maior que 255).
* **Arredondamento:** tanto na quantização quanto na reconstrução final dos pixels, use arredondamento padrão; os casos de teste evitam ambiguidade `.5`.
* **Base ortonormal:** a normalização $\alpha(u)$ e $\alpha(v)$ deve ser aplicada exatamente como especificado — sem ela, a IDCT não reconstrói corretamente.

#### 🧠 Fundamentação Teórica

| Etapa | Análoga no padrão JPEG real | Onde a qualidade é perdida |
|---|---|---|
| Centralização | Mesma — DCT assume sinal centrado em zero | Nenhuma perda |
| DCT-II | Etapa 3–4 do pipeline (@fig-05-jpeg-pipeline) | Nenhuma perda (transformação exata e reversível) |
| Quantização | Etapa 5 — divisão por $Q(u,v)$ | **Principal fonte de perda** — coeficientes de alta frequência viram zero |
| IDCT | Reconstrução final | Reconstrói exatamente os coeficientes *quantizados*, não os originais |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$.
* $N$ linhas seguintes: bloco original $f(x,y)$, inteiros em $[0,255]$.
* $N$ linhas seguintes: tabela de quantização $Q$, inteiros positivos.

**Saída:**

* Bloco reconstruído $f'(x,y)$, $N \times N$, inteiros em $[0,255]$, separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 4<br>120 130 125 128<br>115 140 135 122<br>118 150 160 130<br>110 120 145 138<br>4 6 8 10<br>6 8 10 12<br>8 10 12 16<br>10 12 16 20 | 118 126 119 131<br>114 143 140 119<br>117 149 159 130<br>107 121 146 139 | Após DCT, quantização agressiva nas altas frequências (valores grandes de $Q$ no canto inferior direito) e reconstrução via IDCT, o bloco fica **próximo** do original, mas não idêntico — a diferença é o custo da compressão *lossy*. |

#### 💡 Dica de Depuração

Se o resultado não bater, verifique nesta ordem: (1) os coeficientes DCT brutos (antes da quantização) — eles devem reconstruir o original **exatamente** via IDCT se você pular a etapa 6–7; (2) a tabela $\alpha(u)$ — erro comum é aplicar $\sqrt{2/N}$ também para $u=0$; (3) o arredondamento da quantização, que deve ocorrer **antes** de multiplicar de volta por $Q$.



In [77]:
#| label: fig-05-sim-ep05
#| fig-cap: "Simulador: Pipeline JPEG completo em bloco"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0505" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Pipeline JPEG (bloco 4×4)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🏆 DCT → Q → IDCT</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o fator de qualidade e observe o bloco reconstruído se afastar (ou se aproximar) do original.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#7c3aed;">Escala de Q (1 = tabela base, maior = mais perda)</label>
        <span id="ep0505_vl_s" style="font-family:monospace;font-weight:bold;color:#7c3aed;">1.0×</span>
      </div>
      <input id="ep0505_sl_s" style="width:100%;accent-color:#7c3aed;" max="5" min="0.5" step="0.5" type="range" value="1">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Bloco Original</p>
        <div id="ep0505_grid_o" style="display:grid;grid-template-columns:repeat(4,46px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Reconstruído (DCT→Q→IDCT)</p>
        <div id="ep0505_grid_r" style="display:grid;grid-template-columns:repeat(4,46px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0505_debug" style="margin-top:20px;background:#f3e8ff;border-radius:8px;padding:10px;border:1px solid #e9d5ff;font-family:monospace;font-size:11px;color:#6b21a8;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root) return;
    if(root.dataset.init) return;
    root.dataset.init = "1";
    var N=4;
    var f=[[120,130,125,128],[115,140,135,122],[118,150,160,130],[110,120,145,138]];
    var Qbase=[[4,6,8,10],[6,8,10,12],[8,10,12,16],[10,12,16,20]];
    var s=root.querySelector('#ep0505_sl_s'), sv=root.querySelector('#ep0505_vl_s');
    var go=root.querySelector('#ep0505_grid_o'), gr=root.querySelector('#ep0505_grid_r'), dbg=root.querySelector('#ep0505_debug');

    function alpha(k){ return k===0 ? Math.sqrt(1/N) : Math.sqrt(2/N); }

    function dct2(g){
      var C=[]; for(var u=0;u<N;u++){C.push(new Array(N).fill(0));}
      for(var u=0;u<N;u++){
        for(var v=0;v<N;v++){
          var sum=0;
          for(var x=0;x<N;x++){
            for(var y=0;y<N;y++){
              sum += g[x][y]*Math.cos(Math.PI*(2*x+1)*u/(2*N))*Math.cos(Math.PI*(2*y+1)*v/(2*N));
            }
          }
          C[u][v] = alpha(u)*alpha(v)*sum;
        }
      }
      return C;
    }
    function idct2(C){
      var g=[]; for(var x=0;x<N;x++){g.push(new Array(N).fill(0));}
      for(var x=0;x<N;x++){
        for(var y=0;y<N;y++){
          var sum=0;
          for(var u=0;u<N;u++){
            for(var v=0;v<N;v++){
              sum += alpha(u)*alpha(v)*C[u][v]*Math.cos(Math.PI*(2*x+1)*u/(2*N))*Math.cos(Math.PI*(2*y+1)*v/(2*N));
            }
          }
          g[x][y] = sum;
        }
      }
      return g;
    }
    function cell(v){
      var c=document.createElement('div');
      c.style.cssText='width:46px;height:36px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;background:#f5f3ff;color:#5b21b6;border:1px solid #ddd6fe;';
      c.textContent=v;
      return c;
    }
    function render(){
      var scale=parseFloat(s.value);
      sv.textContent=scale.toFixed(2)+'×';
      go.innerHTML=''; gr.innerHTML='';
      var g=[]; for(var x=0;x<N;x++){ var row=[]; for(var y=0;y<N;y++){ row.push(f[x][y]-128); } g.push(row); }
      var C=dct2(g);
      var Cq=[]; for(var u=0;u<N;u++){ var row=[]; for(var v=0;v<N;v++){ var Qv=Qbase[u][v]*scale; var q=Math.round(C[u][v]/Qv); row.push(q*Qv); } Cq.push(row); }
      var gr2=idct2(Cq);
      var diffSum=0, n=0;
      for(var x=0;x<N;x++){
        for(var y=0;y<N;y++){
          go.appendChild(cell(f[x][y]));
          var rec = Math.round(gr2[x][y]+128);
          rec = Math.max(0, Math.min(255, rec));
          gr.appendChild(cell(rec));
          diffSum += Math.abs(rec-f[x][y]); n++;
        }
      }
      dbg.textContent = 'Erro médio absoluto por pixel: ' + (diffSum/n).toFixed(2) + '  |  Quanto maior a escala de Q, maior o erro de reconstrução.';
    }
    s.addEventListener('input', render);
    render();
  }
  function tryInit(){ var root=document.getElementById('sim-ep0505'); if(root) init(root); else setTimeout(tryInit,200); }
  tryInit();
})();
</script>
''')

In [78]:
%%writefile EP05_05.py
# Código Python

Writing EP05_05.py


In [79]:
TestSuite("EP05_05.py").run()